In [1]:
## important: on simule un utilsateur qui navigue sur agatha
## 
## le fonctionnement est le suivant: on doit faire une requete http pour charger
## la page et (surtout) pour récuperer les cookies de session qui vont bien et 
## qui stockent notre requete. Ensuite on peut faire appel au webservice "ajax.php"
## qui permet d'effectivement récupérer les informations qu'on cherche dans les
## databases des archives.
##
## Pour bien piger comment ca marche, il est utile d'aller jeter un oeil au notebook
## qui s'appelle "en_mode_gros_bourrin" car c'est ici que j'ai enregistré les infos
## utiles (tq un prototype de schéma json pour les requetes et reponses).

from typing import Annotated, Literal
import pydantic
import requests as rq
from textdistance import levenshtein

In [32]:
BASE_URL  = "https://agatha.arch.be/search"
AJAX_URL  = f"{BASE_URL}/genealogie/ajax.php"
GENEA_URL = f"{BASE_URL}/genealogie/results/"

## 
class Municipality(pydantic.BaseModel):
    """All the technical data used to retrieve a given municipality"""

    label:         Annotated[str, "How this municipality should be displayed"]
    value:         Annotated[str, "Name that should be used when referencing this municipality"]
    CURRENT_NAME:  Annotated[str, "The current denomincation of this municipality"]
    PLACEPOINT_ID: Annotated[int, "The effective identifier of this municipality"]
    PARISHES_ID:   Annotated[int | None, "If present, the id of the parish in the municipality"]

def municipality_by_name(
    name:   str, 
) -> list[Municipality]:
    """Find municipality data by name."""
    r = rq.post(AJAX_URL, data = {
        "method": "getMunicipalities",
        "term"  : name,   # location qu'on veut trouver
    })
    return [ Municipality(**x) for x in r.json() ]

## 
GenealogySource = Literal[
    Annotated["PR", "Church Register before 1796"],
    Annotated["CS", "Civil State after 1796"]
]

## Types relating to a genealogical search within church registers
ParishActType = Literal[
    Annotated["1",     "Baptism/Birth"],
    Annotated["2",     "Weddings"],
    Annotated["3",     "Funeral/Deaths"],
    Annotated["OTHER", "Other type of documents"]
]
OtherParishActType = Literal[
    Annotated["11",    "List of communicants (Persons who have received the sacrament of eucharist.)"],
    Annotated["12",    "Certificates of dispensations of marriage (Certificate by which an authority releases a person from an obligation or prohibition, in this case the celebration of marriage.)"],
    Annotated["20",    "Certificates of impediment of marriage (Certificate of objection to the marriage on the grounds of non-compliance with a condition for the celebration of the marriage.)"],
    Annotated["23",    "List of names (List of person drawn up for an event (e.g. for Sunday prayers) or for an association (e.g. membership list of a fraternity)."],
    Annotated["31",    "List of confirmands (Persons who have received the sacrament of confirmation.)"],
    Annotated["56",    "Memory masses (List of deceased persons for whom a church service should be organised annually in memory and for the salvation of souls.)"],
    Annotated["28",    "Status animarum (List per street or district of the families in a parish (state of souls), with information about the age of the residents, confirmation, ...)"],
]
ParishIndices = Literal[
    Annotated["1",     "Baptism/Birth"],
    Annotated["2",     "Weddings"],
    Annotated["3",     "Funeral/Deaths"],
]


## Types relating to a genealogical search within civil registers
CivilActType = Literal[
    Annotated["1",     "Birth"],
    Annotated["2",     "Marriages"],
    Annotated["3",     "Deaths"],
    Annotated["OTHER", "Other type of documents"]
]
OtherCivilActType = Literal[
    Annotated["BS_U",  "Additional register (Supplement to a register containing exhibited documents (extracts) or notes.)"],
    Annotated["BS_A",  "Adoption certificates (Certificate by which a court or authority establishes a legal direct relationship between persons who are not blood relatives.)"],
    Annotated["BS_Ob", "Death annexes (Extracts of deaths abroad or death declarations of missing persons (after the World Wars) attached to death certificates.)"],
    Annotated["BS_E",  "Divorce certificates (Certificate of the termination of marriage.)."],
    Annotated["BS_N",  "Naturalisation certificates (Certificate by which a court or authority grants nationality to a person)."],
]
CivilIndices = Literal[
    Annotated["BS_Ei",     "Annual index of divorce certificates"],
    Annotated["BS_Ni",     "Annual index of naturalisation certificates"],
    Annotated["BS_Ui",     "Index of additional register"],
    Annotated["BS_Ax",     "Ten-year index of adoption certificates"],
    Annotated["BS_Ex",     "Ten-year index of divorce certificates"],
    Annotated["BS_Nx",     "Ten-year index of naturalisation certificates"],
]

class Parish(pydantic.BaseModel):
    """A parish identified within a community"""

    PARISH_ID:    Annotated[int, "The technical identifier used to reference the parish"]
    PLACE_NAME:   Annotated[str, "The name of the parish"]


def parish_by_municipality(
    municipality: Municipality,
) -> list[Parish]:
    """Find the list of parishes within a given municipality."""
    r = rq.post(AJAX_URL, data = {
        "method":        "getParishFromMunicipality",
        "PLACEPOINT_ID": municipality.PLACEPOINT_ID,
    })
    return [ Parish(**x) for x in r.json() ]


In [33]:


class GenealogySearchResult(pydantic.BaseModel): 
    """A simplified interface that yields all the useful information (and no more)"""

    description: Annotated[str, "Description"]
    start_date:  Annotated[str, "First day covered by this document"]
    end_date:    Annotated[str, "Last day covered by this document"]
    last_update: Annotated[str, "Date when this archive was last modified"]
    notes:       Annotated[str, "Remarks and notes about this document"]
    available:   Annotated[bool,"Is this document available for online consultation"]
    url:         Annotated[str | None, "URL where this document can be consulted (if available)"]
    

class GenealogyDocument(pydantic.BaseModel):
    """A document that has been found by a research in the genealogic sources."""
    
    BOOKPART_ID: Annotated[str, "Identifier that is used to retrieve the actual document scan"]
    BOOKPARTPLACE_ID: Annotated[str, "I don't know what this means"]
    NEW_MUNICIPALITIES: Annotated[str | None, "New municipality name (after merge)"]
    OLD_MUNICIPALITIES: Annotated[str | None, "Municipality name (before they were merged)"]
    PLACE_NAME: Annotated[str | None, "Parish name (if relevant)"]
    FULL_PARISH_NAME: Annotated[str | None, "Full parish name (if relevant)"]
    START_DAY: Annotated[int | None, "First day covered by this document (day)"]
    START_MONTH: Annotated[int | None, "First day covered by this document (month)"]
    START_YEAR: Annotated[int | None, "First day covered by this document (year)"]
    END_DAY: Annotated[int | None, "Last day covered by this document (day)"]
    END_MONTH: Annotated[int | None, "Last day covered by this document (month)"]
    END_YEAR: Annotated[int | None, "Last day covered by this document (year)"]
    BOOK_LAST_MODIF_DATE: Annotated[str | None, "Date when this archive was last modified"]
    DESCRIPTION: Annotated[str | None, "Description"]
    FULL_REMARKS: Annotated[str | None, "Remarks wrt the document (e.g. missing)"]
    PUBLISH_DATE_REF: Annotated[str | None, "Date when this archive was published"]
    EDITORIALFORMBASE_ID: Annotated[str | None, "Identifier of the document type"]
    ISINDEX: Annotated[str | None, "Does it come from an index"]
    NOTES: Annotated[str | None, "Additional notes regarding this document"]
    AVAILABLE: Annotated[bool | None, "Is the document available for online consultation"]

    def simplified(self) -> GenealogySearchResult:
        """Returns a simplified version of this object"""
        return GenealogySearchResult(
            description = self.DESCRIPTION,
            start_date  = f"{self.START_DAY:02d}/{self.START_MONTH:02d}/{self.START_YEAR:02d}",
            end_date    = f"{self.END_DAY:02d}/{self.END_MONTH:02d}/{self.END_YEAR:02d}",
            last_update = self.BOOK_LAST_MODIF_DATE,
            notes       = (self.NOTES + "\n" if self.NOTES is not None else "") + (self.FULL_REMARKS + "\n" if self.FULL_REMARKS is not None else ""),
            available   = self.AVAILABLE,
            url         = self.find_url()
        )

    def find_url(self) -> str:
        """Retrieve this document URL"""
        if self.AVAILABLE:
            return rq.get(f"{BASE_URL}/genealogie/{self.BOOKPART_ID}").url
            #return f"{BASE_URL}/genealogie/{self.BOOKPART_ID}"
        else:
            return None

def genealogy_search_church_registers_prior_to_1796(
    municipality_name: Annotated[str, "Name of the municipality"],
    parish_name: Annotated[str | None, "Name of the parish"] = None,
    act_types: Annotated[list[ParishActType], "The kind of document we are searching for"] = None,
    other_types: Annotated[list[OtherParishActType] | None, "Mandatory when act_types is OTHER"] = None,
    index_types: Annotated[list[ParishIndices] | None, "The kind of index where to search data"] = None,
    start_year: Annotated[int | None, "Start year of the considered period"] = None,
    end_year: Annotated[int | None, "End year of the considered period"] = None,
) -> list[GenealogyDocument]:
    """Search in the Church Registers (prior to 1796).

    Note: providing at least one 'act_types' or 'index_types' in mandatory.
    """
    municipality = min(municipality_by_name(municipality_name), key = lambda x: levenshtein(x.value, municipality_name))
    parishes     = parish_by_municipality(municipality)

    if parish_name is not None:
        parishes = [ p for p in parishes if parish_name in p.PLACE_NAME.lower() ]
        
    #
    request_data = {
        "WHICH_TAB": "PR",
        "PLACEPOINT_ID_PR" : municipality.PLACEPOINT_ID,
        "PLACE_PR": municipality.value,
        "CURRENT_NAME_PR": municipality.CURRENT_NAME,
        "PARISH_ID[]": ",".join( str(p.PARISH_ID) for p in parishes ),
    }
    
    if act_types is not None:
        request_data["TYPEACTES_PR[]"] = ",".join(act_types)
    
    if other_types is not None:
        request_data["TYPEACTES_OTHER_PR[]"] = other_types

    if index_types is not None:
        request_data["TYPEINDEX_PR[]"] = index_type

    if start_year is not None:
        request_data["exactYear"] = start_year

    if end_year is not None:
        request_data["SHOW_PERIODE"] = end_year
    
    ## search genea
    search  = rq.post(GENEA_URL, data = request_data)
    cookies = dict(search.cookies)
    ## fetch results
    results = rq.post(AJAX_URL, 
                      data = {
                          "method": "getResults", 
                          "pageSize": 10, 
                          "pageNumber": 1,
                      },
                      cookies = cookies)

    results = results.json()
    results = results["response"]["docs"]
    return [GenealogyDocument(**x).simplified() for x in results]



def genealogy_search_civil_state_registers(
    municipality_name: Annotated[str, "Name of the municipality"],
    act_types: Annotated[list[CivilActType], "The kind of document we are searching for"] = None,
    other_types: Annotated[list[OtherCivilActType] | None, "Mandatory when act_types is OTHER"] = None,
    index_types: Annotated[list[CivilIndices] | None, "The kind of index where to search data"] = None,
    start_year: Annotated[int | None, "Start year of the considered period"] = None,
    end_year: Annotated[int | None, "End year of the considered period"] = None,
) -> list[GenealogyDocument]:
    """Search in the Church Registers (prior to 1796).

    Note: providing at least one 'act_types' or 'index_types' in mandatory.
    """
    municipality = min(municipality_by_name(municipality_name), key = lambda x: levenshtein(x.value, municipality_name))

    request_data = {
        "WHICH_TAB": "CS",
        "PLACEPOINT_ID_CS" : municipality.PLACEPOINT_ID,
        "PLACE_CS": municipality.value,
        "CURRENT_NAME_CS": municipality.CURRENT_NAME,
        "CHOOSE_DATE_PERIODE": "1",
    }
    
    if act_types is not None:
        request_data["TYPEACTES_CS[]"] = ",".join(act_types)
    
    if other_types is not None:
        request_data["TYPEACTES_OTHER_CS[]"] = other_types

    if index_types is not None:
        request_data["TYPEINDEX_CS[]"] = index_type

    if start_year is not None:
        request_data["exactYear"] = start_year

    if end_year is not None:
        request_data["SHOW_PERIODE"] = end_year
    
    ## search genea
    search  = rq.post(GENEA_URL, data = request_data)
    cookies = dict(search.cookies)
    ## fetch results
    results = rq.post(AJAX_URL, 
                      data = {
                          "method": "getResults", 
                          "pageSize": 10, 
                          "pageNumber": 1,
                      },
                      cookies = cookies)

    results = results.json()
    results = results["response"]["docs"]
    return [GenealogyDocument(**x).simplified() for x in results]
    

In [34]:
xx = genealogy_search_church_registers(
    "sterrebeek", 
    act_types=["3"],
)

for x in xx:
    print(x.model_dump_json(indent=4))
    print()

{
    "description": "Funeral and death certificates",
    "start_date": "22/10/1651",
    "end_date": "12/12/1778",
    "last_update": "12/02/2020",
    "notes": "\n",
    "available": true,
    "url": "https://agatha.arch.be/data/images/518/518_0146_000_00002_000/0_0106"
}

{
    "description": "Funeral and death certificates",
    "start_date": "05/02/1803",
    "end_date": "18/07/1803",
    "last_update": "12/02/2020",
    "notes": "\n",
    "available": true,
    "url": "https://agatha.arch.be/data/images/518/518_0146_000_00004_000/0_0067"
}

{
    "description": "Funeral and death certificates",
    "start_date": "05/01/1802",
    "end_date": "22/11/1802",
    "last_update": "12/02/2020",
    "notes": "\n",
    "available": true,
    "url": "https://agatha.arch.be/data/images/518/518_0146_000_00004_000/0_0060"
}

{
    "description": "Funeral and death certificates",
    "start_date": "19/01/1801",
    "end_date": "08/10/1801",
    "last_update": "12/02/2020",
    "notes": "\n",


In [35]:
xx = genealogy_search_civil_state_registers(
    "sterrebeek", 
    act_types=["3"],
    start_year="1797",
    end_year="2025",
)

for x in xx:
    print(x.model_dump_json(indent=4))
    print()

{
    "description": "Mixed certificates",
    "start_date": "01/01/1891",
    "end_date": "31/12/1895",
    "last_update": "19/01/2022",
    "notes": "",
    "available": true,
    "url": "https://agatha.arch.be/data/images/518/518_9999_999_2375844_000/0_0440"
}

{
    "description": "Mixed certificates",
    "start_date": "01/01/1896",
    "end_date": "31/12/1902",
    "last_update": "19/01/2022",
    "notes": "",
    "available": true,
    "url": "https://agatha.arch.be/data/images/518/518_9999_999_2375845_000/0_0005"
}

{
    "description": "Mixed certificates",
    "start_date": "01/01/1903",
    "end_date": "31/12/1910",
    "last_update": "19/01/2022",
    "notes": "",
    "available": true,
    "url": "https://agatha.arch.be/data/images/518/518_9999_999_2375846_000/0_0005"
}

{
    "description": "Death certificates",
    "start_date": "01/01/1866",
    "end_date": "31/12/1866",
    "last_update": "19/01/2022",
    "notes": "",
    "available": true,
    "url": "https://agatha.